# Ahrefs growth test: HUM Nutrition (Domain Rating fallback)

The current Ahrefs API key (scope: "API v3 for public endpoints") only has access to Ahrefs' free public endpoints, not the paid Site Explorer endpoints this notebook originally used for organic/paid traffic history — calling those returns `401 Unauthorized`.

This version falls back to the one metric the free tier does expose: **Domain Rating** (backlink authority, 0-100) via `/v3/public/domain-rating-free`. That endpoint has no history lookup — it always returns today's value regardless of any date filter — so growth here is tracked by re-running this notebook over time and comparing snapshots saved locally to `dr_history.csv`. It won't show meaningful growth until it's been run on at least two different days.

If/when the Ahrefs plan gets full Site Explorer API v3 access, swap back to `/v3/site-explorer/metrics` and `/v3/site-explorer/metrics-history` for real organic/paid traffic growth.

In [1]:
from __future__ import annotations

import csv
import os
from datetime import date
from pathlib import Path

import requests

TARGET = "humnutrition.com"
BASE_URL = "https://api.ahrefs.com/v3/public"
HISTORY_FILE = Path("dr_history.csv")


def load_env_value(*names: str) -> str:
    for name in names:
        value = os.getenv(name, "").strip()
        if value:
            return value

    env_path = next(
        (parent / ".env" for parent in [Path.cwd(), *Path.cwd().parents] if (parent / ".env").exists()),
        None,
    )
    if env_path is not None:
        for raw_line in env_path.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, value = line.split("=", 1)
            if key.strip() in names:
                cleaned = value.strip().strip('"').strip("'")
                if cleaned:
                    return cleaned

    return ""


API_KEY = load_env_value("ahrefs_api_key", "AHREFS_API_KEY")

if not API_KEY:
    raise SystemExit("Set ahrefs_api_key or AHREFS_API_KEY in .env before running this notebook.")

HEADERS = {
    "Authorization": f"Bearer {API_KEY}",
    "Accept": "application/json",
}


def ahrefs_public_get(endpoint: str, params: dict[str, str]) -> dict:
    response = requests.get(f"{BASE_URL}/{endpoint}", headers=HEADERS, params=params, timeout=60)
    response.raise_for_status()
    return response.json()


def safe_pct(current: float | int | None, previous: float | int | None) -> float | None:
    if previous in (None, 0):
        return None
    if current is None:
        return None
    return ((float(current) - float(previous)) / float(previous)) * 100.0


today = date.today()

print(f"Loaded Ahrefs key: {'yes' if API_KEY else 'no'}")
print(f"Target: {TARGET}")
print(f"Today: {today.isoformat()}")

Loaded Ahrefs key: yes
Target: humnutrition.com
Today: 2026-09-03


In [2]:
# Snapshot: current Ahrefs Domain Rating (free public endpoint).
# Note: unlike the paid Site Explorer endpoints, this always returns TODAY's
# value regardless of any date filter -- there is no historical lookup on the
# free tier, so growth can only be computed by re-running this notebook on
# different days and comparing snapshots saved locally in dr_history.csv.
dr_response = ahrefs_public_get(
    "domain-rating-free",
    {"target": TARGET, "date": today.isoformat()},
)

current_dr = dr_response["domain_rating"]["domain_rating"]
print(f"{TARGET} Domain Rating today: {current_dr}")

humnutrition.com Domain Rating today: 68.0


In [3]:
# Append today's snapshot to the local history log (one row per day; reruns on
# the same day overwrite that day's row instead of duplicating it), then
# compute growth against the most recent prior snapshot and the earliest one
# on record.
rows: dict[str, float] = {}
if HISTORY_FILE.exists():
    with HISTORY_FILE.open(newline="", encoding="utf-8") as fh:
        for row in csv.DictReader(fh):
            rows[row["date"]] = float(row["domain_rating"])

rows[today.isoformat()] = current_dr

with HISTORY_FILE.open("w", newline="", encoding="utf-8") as fh:
    writer = csv.writer(fh)
    writer.writerow(["date", "domain_rating"])
    for d in sorted(rows):
        writer.writerow([d, rows[d]])

history = sorted(rows.items())  # [(date_str, dr), ...] oldest first
earliest_date, earliest_dr = history[0]
previous_entries = [entry for entry in history if entry[0] != today.isoformat()]
previous_date, previous_dr = previous_entries[-1] if previous_entries else (None, None)

growth_since_previous_run = safe_pct(current_dr, previous_dr)
growth_since_first_snapshot = (
    safe_pct(current_dr, earliest_dr) if earliest_date != today.isoformat() else None
)

summary = {
    "snapshots_recorded": len(history),
    "earliest_date": earliest_date,
    "earliest_dr": earliest_dr,
    "previous_date": previous_date,
    "previous_dr": previous_dr,
    "today_date": today.isoformat(),
    "today_dr": current_dr,
    "growth_since_previous_run_pct": round(growth_since_previous_run, 2)
    if growth_since_previous_run is not None
    else None,
    "growth_since_first_snapshot_pct": round(growth_since_first_snapshot, 2)
    if growth_since_first_snapshot is not None
    else None,
}

summary

{'snapshots_recorded': 1,
 'earliest_date': '2026-09-03',
 'earliest_dr': 68.0,
 'previous_date': None,
 'previous_dr': None,
 'today_date': '2026-09-03',
 'today_dr': 68.0,
 'growth_since_previous_run_pct': None,
 'growth_since_first_snapshot_pct': None}

In [4]:
# Compact readout for the drill-down definition.
print(f"HUM Nutrition Domain Rating today: {summary['today_dr']}")
if summary["previous_dr"] is not None:
    print(f"Change since last snapshot ({summary['previous_date']}): {summary['growth_since_previous_run_pct']}%")
else:
    print("No prior snapshot yet -- rerun this notebook on a later date to start tracking growth.")
print(f"Snapshots recorded so far: {summary['snapshots_recorded']} (saved to {HISTORY_FILE})")
print()
print("Note: Domain Rating measures backlink profile strength (0-100), NOT search")
print("traffic. It's a proxy for authority/growth, not a substitute for the organic")
print("traffic metric this notebook originally targeted -- that requires paid Ahrefs")
print("Site Explorer API v3 access, which the current key's scope doesn't include.")

HUM Nutrition Domain Rating today: 68.0
No prior snapshot yet -- rerun this notebook on a later date to start tracking growth.
Snapshots recorded so far: 1 (saved to dr_history.csv)

Note: Domain Rating measures backlink profile strength (0-100), NOT search
traffic. It's a proxy for authority/growth, not a substitute for the organic
traffic metric this notebook originally targeted -- that requires paid Ahrefs
Site Explorer API v3 access, which the current key's scope doesn't include.
